# [실습 10] 한국어 ASR 모방학습 — Whisper × Zeroth-Korean

**목표** — 음성 도메인에서의 **모방학습(SFT)** 을 직접 손으로 돌려봅니다.  
OpenAI Whisper 의 작은 모델에 한국어 음성 인식 데이터셋을 학습시켜, 음성 → 한글 텍스트 인식 정확도(WER)를 끌어올립니다.

## 강화학습 관점에서의 ASR 모방학습
- 상태 $s$ = 음성 mel-spectrogram + 지금까지 디코딩한 토큰열
- 행동 $a$ = 다음에 출력할 토큰
- 시범 $a^*$ = 정답 전사(transcript)의 다음 토큰
- 손실 $\mathcal{L} = -\log \pi_\theta(a^*|s)$ — teacher forcing cross-entropy

즉, ASR 의 표준 학습은 **사람이 만든 전사를 "전문가 시범" 으로 그대로 따라하는 Behavioral Cloning** 입니다.

| 항목 | 값 |
|---|---|
| 데이터셋 | [`kresnik/zeroth_korean`](https://huggingface.co/datasets/kresnik/zeroth_korean) — 한국어 ASR 약 51시간 |
| 모델 | [`openai/whisper-tiny`](https://huggingface.co/openai/whisper-tiny) — 39M, Encoder-Decoder Transformer |
| 평가 지표 | WER (Word Error Rate), CER (Character Error Rate) |
| 권장 환경 | Colab T4 GPU |

## 0. 환경 설치

In [ ]:
!pip install -q --upgrade \
    transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    librosa soundfile jiwer evaluate

In [ ]:
import os, torch
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

---
## 1. 데이터셋 — Zeroth-Korean

한국어 ASR 공개 벤치마크. 16kHz 발화 + 정답 전사 한 쌍이 들어있습니다.

In [ ]:
from datasets import load_dataset, Audio

# 학습 속도를 위해 일부만 — 전체로 돌릴 때는 split="train" 그대로
train_ds = load_dataset("kresnik/zeroth_korean", split="train[:2000]")
test_ds  = load_dataset("kresnik/zeroth_korean", split="test[:200]")

# Whisper 입력 규격에 맞춰 16kHz 로 리샘플링
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds  = test_ds.cast_column("audio", Audio(sampling_rate=16000))

print("train 샘플:", len(train_ds), "| test 샘플:", len(test_ds))
print("컬럼     :", train_ds.column_names)
print("\n첫 샘플:")
print(" - text :", train_ds[0]["text"])
print(" - audio:", {k: type(v).__name__ for k, v in train_ds[0]["audio"].items()})
print(" - 길이 :", len(train_ds[0]["audio"]["array"]) / 16000, "초")

In [ ]:
# 음성 직접 들어보기 (Colab/IPython 한정)
from IPython.display import Audio as AudioWidget, display
import numpy as np

for i in range(2):
    s = train_ds[i]
    print(f"[{i}] {s['text']}")
    display(AudioWidget(s["audio"]["array"], rate=16000))

In [ ]:
# 발화 길이 분포 — Whisper 가 30초 chunk 단위로 처리하므로 30초 이하인지 확인
import numpy as np
durations = np.array([len(s["array"]) / s["sampling_rate"] for s in train_ds["audio"]])
print(f"평균 {durations.mean():.1f}s | p95 {np.percentile(durations,95):.1f}s | max {durations.max():.1f}s")

---
## 2. Whisper Processor & 모델 로드

Whisper 는 **Feature Extractor** (mel-spectrogram 변환) + **Tokenizer** (다국어 BPE) 두 가지를 함께 다루는 **Processor** 를 제공합니다.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="korean", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to("cuda" if torch.cuda.is_available() else "cpu")

# 한국어 디코딩으로 강제 설정
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print("파라미터 수:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

### 학습 전 베이스라인 WER

사전학습된 Whisper-tiny 만으로 한국어를 얼마나 인식하는지 먼저 측정합니다.

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

@torch.no_grad()
def transcribe_batch(batch_samples, model_, processor_):
    arrays = [s["audio"]["array"] for s in batch_samples]
    inp = processor_.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt").to(model_.device)
    pred_ids = model_.generate(input_features=inp.input_features, max_new_tokens=128)
    preds = processor_.batch_decode(pred_ids, skip_special_tokens=True)
    refs  = [s["text"] for s in batch_samples]
    return preds, refs

def evaluate_wer(eval_ds, model_, processor_, batch_size=8, n_samples=None):
    preds_all, refs_all = [], []
    n = len(eval_ds) if n_samples is None else min(n_samples, len(eval_ds))
    for i in range(0, n, batch_size):
        batch = [eval_ds[j] for j in range(i, min(i + batch_size, n))]
        p, r = transcribe_batch(batch, model_, processor_)
        preds_all.extend(p); refs_all.extend(r)
    wer = wer_metric.compute(predictions=preds_all, references=refs_all)
    cer = cer_metric.compute(predictions=preds_all, references=refs_all)
    return wer, cer, preds_all, refs_all

wer_b, cer_b, preds_b, refs_b = evaluate_wer(test_ds, model, processor, n_samples=80)
print(f"[BEFORE] WER = {wer_b*100:.2f}%  | CER = {cer_b*100:.2f}%")
for i in range(3):
    print(f"\n GT  : {refs_b[i]}")
    print(f" PRED: {preds_b[i]}")

---
## 3. 전처리 — 오디오 → log-mel, 텍스트 → token id

`prepare_dataset` 함수가 한 샘플에서:
- `input_features` : 80×3000 log-mel spectrogram 텐서
- `labels` : 정답 transcript 의 토큰 id 시퀀스 (디코더 입력/예측 대상)

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_proc = train_ds.map(prepare_dataset,
                          remove_columns=train_ds.column_names,
                          num_proc=1)
test_proc  = test_ds.map(prepare_dataset,
                          remove_columns=test_ds.column_names,
                          num_proc=1)
print("전처리 완료:", train_proc, test_proc, sep="\n")

### Data Collator — 가변 길이 시퀀스 패딩

오디오는 이미 30초로 맞춰져 있지만, **레이블 길이는 가변**이라 직접 패딩 콜레이터를 정의합니다.  
패딩 토큰 자리는 loss 계산에서 제외하기 위해 `-100` 으로 마스킹합니다.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # 1) 입력(오디오 feature) 처리
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2) 레이블(텍스트) 처리 + 패딩 토큰을 -100 으로
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # BOS 가 이미 들어가 있으면 제거 (Whisper 가 디코더 입력에서 자동 prepend)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = WhisperDataCollator(processor=processor)
print("collator OK")

---
## 4. Seq2SeqTrainer 로 모방학습 실행

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# 메모리 절약 옵션
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.gradient_checkpointing_enable()
model.config.use_cache = False

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # -100 → pad_token_id 로 복원 후 디코딩
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
    }

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_ko_sft",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=20,
    max_steps=200,                     # 데모용 — 실제는 1000+
    fp16=True,
    predict_with_generate=True,
    generation_max_length=128,
    evaluation_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=8,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_proc,
    eval_dataset=test_proc,
    data_collator=collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)
print("Trainer 준비 완료")

In [ ]:
trainer.train()
trainer.save_model("./whisper_ko_sft_final")
processor.save_pretrained("./whisper_ko_sft_final")
print("모방학습 완료 — ./whisper_ko_sft_final 에 저장")

---
## 5. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history

def pull(key):
    xs, ys = [], []
    for h in hist:
        if key in h and "step" in h:
            xs.append(h["step"]); ys.append(h[key])
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))

x, y = pull("loss")
if x: axes[0].plot(x, y, marker="o", linewidth=1)
axes[0].set_title("Training Loss"); axes[0].grid(alpha=0.3)

xw, yw = pull("eval_wer")
xc, yc = pull("eval_cer")
if xw: axes[1].plot(xw, [v*100 for v in yw], marker="o", label="WER (%)")
if xc: axes[1].plot(xc, [v*100 for v in yc], marker="s", label="CER (%)")
axes[1].set_title("Eval Error Rate"); axes[1].legend(); axes[1].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("step")
plt.tight_layout(); plt.show()

---
## 6. 학습 전/후 응답 비교

In [ ]:
wer_a, cer_a, preds_a, refs_a = evaluate_wer(test_ds, model, processor, n_samples=80)
print(f"[BEFORE] WER = {wer_b*100:.2f}%  | CER = {cer_b*100:.2f}%")
print(f"[AFTER ] WER = {wer_a*100:.2f}%  | CER = {cer_a*100:.2f}%")
print(f"        \u0394WER = {(wer_a-wer_b)*100:+.2f}%p")
print("-" * 60)
for i in range(5):
    print(f"GT    : {refs_a[i]}")
    print(f"BEFORE: {preds_b[i]}")
    print(f"AFTER : {preds_a[i]}")
    print()

---
## 7. 정리 & 다음 단계

- ✅ 음성 도메인의 모방학습 = **(음성, 정답 전사)** 시범 데이터를 그대로 따라하는 CE 학습
- ✅ 200스텝 데모에서도 한국어 WER 이 큰 폭으로 떨어짐을 확인
- ⚠️ Whisper-tiny 는 표현력이 작아 한계가 있음 → `whisper-base` / `whisper-small` 로 갈수록 빠르게 개선

### 한 걸음 더
- 🔧 `max_steps=1000`+, `learning_rate=5e-6`, 모델을 `whisper-base` 로 바꿔 본격 학습
- 🔧 다음 실습 [[실습11] 강화학습 (WER reward REINFORCE)] 로 이어가 **모방학습의 한계인 exposure bias** 를 보상신호로 보완

### 강화학습으로 가는 동기 — Exposure Bias
모방학습은 **항상 정답 토큰을 보고 다음 토큰을 예측** 합니다(teacher forcing).  
→ 추론 시에는 자기가 만든 토큰을 보고 다음을 예측해야 하는데, 학습/추론 분포가 어긋남.  
→ 한 번 틀리면 그 뒤로 누적 오류가 폭증.  
→ **시퀀스 단위 보상(WER)** 으로 직접 정책을 갱신하는 RL 이 이 문제를 줄여줍니다 ➡ 다음 노트북.